# router-voto-r3 — reentreno del voto con el dataset combinado (X87)

**Qué es:** el mismo fine-tune del notebook `router-voto` (layaft sobre el base
multilingual de `bmo-laya-base`, encoder incluido), ahora con `data=voto_entrenamiento.jsonl`:
**858 casos** = r2 verificado (558, byte a byte, del dataset `flow-router-voto`) + 300 casos
congelados del repo deepseek-flow (texts con sha256 `2947b449…`, etiqueta final de la persona).
Anti-fuga del combinado contra bench + control: 0 fugas (umbral Jaccard 0,6).
Archivo con sha256 `4037fe4f246a71ea7c223bf854d20e711e684e5843d152165be64b71989e93aa`,
fijado en `docs/bitacora.md` de deepseek-flow (paso 5, commit 624c259).

**Cómo se preparó el dato (en Kaggle):** subir `voto_entrenamiento.jsonl` como **nueva versión
del dataset `flow-router-voto`** (botón *New Version*, drag & drop; queda junto al
`router_voto_verified.jsonl` original). Este notebook lo copia del input y entrena sobre él.

**Acelerador:** GPU T4 · **Internet:** sí. El test held-out de 30 (jamás entrenado)
sigue siendo la medición interna; la puerta de aceptación real es el bench local de
deepseek-flow (`bench_voto.py --entrenamiento`), con el control fuera de muestra.

In [ ]:
!nvidia-smi
%pip install -q laya-finetune==0.1.2 pyyaml
import layaft, laya, torch
print("layaft listo | laya", laya.__version__, "| torch", torch.__version__, "| GPUs:", torch.cuda.device_count())

In [ ]:
import glob, os, shutil, pathlib
W = pathlib.Path("/kaggle/working")
for name in ("tasks/router_voto.yaml", "tasks/contexts_router.txt", "tasks/router_voto_test.jsonl",
             "data/voto_entrenamiento.jsonl", "data/router_voto_verified.jsonl"):
    hits = glob.glob(f"/kaggle/input/**/{os.path.basename(name)}", recursive=True)
    assert hits, f"no encontré {name}: fijate que el dataset flow-router-voto (con voto_entrenamiento.jsonl) esté como Input"
    dest = W / name
    dest.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(hits[0], dest)
print("listo:", sorted(str(p.relative_to(W)) for p in W.rglob("*") if p.is_file()))

In [ ]:
# Dónde está el base: el directorio de input que contiene model.safetensors (no confiar en el slug de mount)
import hashlib, pathlib
DATO = "/kaggle/working/data/voto_entrenamiento.jsonl"
sha = hashlib.sha256(open(DATO, "rb").read()).hexdigest()
assert sha == "4037fe4f246a71ea7c223bf854d20e711e684e5843d152165be64b71989e93aa", f"sha del dataset no casa: {sha}"
print("dataset OK:", sha[:12], "—", sum(1 for _ in open(DATO)), "casos")
candidatos = [p for p in pathlib.Path("/kaggle/input").rglob("model.safetensors")]
print("inputs:", [str(p) for p in pathlib.Path("/kaggle/input").glob("*")])
assert candidatos, "no encontré el checkpoint base: revisá que bmo-laya-base esté como Input"
BASE = str(candidatos[0].parent)
print("base:", BASE)
# Entrenamiento desde el base local, sobre el COMBINADO; ctx por defecto 1k -> runs/router_voto-1k
!layaft train task=tasks/router_voto.yaml data=data/voto_entrenamiento.jsonl model={BASE} profile=full

In [ ]:
# Validación en el test set escrito a mano (el número que cuenta)
import pathlib
runs = sorted(p for p in pathlib.Path("runs").glob("*"))
print("runs:", [str(r) for r in runs])
RUN = str(runs[0])
!layaft val task=tasks/router_voto.yaml model={RUN}

In [ ]:
import glob, pathlib
for f in sorted(pathlib.Path("runs/val").rglob("*")):
    if f.is_file():
        print("=" * 20, f, "=" * 20)
        print(f.read_text(encoding="utf-8")[:2000])

## Para traerlo a casa

El checkpoint queda en `runs/router_voto-1k/` (pesos + tokenizer + config con temperatura
calibrada). En la máquina local:

```
kaggle kernels output roquedb/router-voto-r3 -p .modelos/router_voto_r3/
```

y el gate de deepseek-flow (las seis condiciones, pre-registro X87):

```
LAYA_MODEL_VOTO=<ruta>/runs/router_voto-1k python3 banco/probes/bench_voto.py --entrenamiento salidas/evals/voto_entrenamiento.jsonl
```